<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/Conditions_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!wget --cache=off -q -O notebook_lib/condition_explorer.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/condition_explorer.py

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
from notebook_lib.condition_explorer import make_condition_explorer
import duckdb
import pandas as pd
from pathlib import Path


In [40]:
# @title Easy Exercise 1
df_easy_ex1 = pd.DataFrame([
    # TRUE rows
    {"age": 25, "country": "Spain", "active": True},
    {"age": 17, "country": "France", "active": True},
    {"age": 30, "country": "Germany", "active": True},

    # FALSE rows
    {"age": 16, "country": "Spain", "active": True},
    {"age": 40, "country": "France", "active": False},
    {"age": 15, "country": "Italy", "active": True},
])

make_condition_explorer(
    explorer_id="easy_1",
    title="Easy Exercise 1",
    subtitle="""
Statement

Select users who:

- are active,
- and are either at least 18 years old or from France.

Exclude users from Italy.
""",
    default_variables=["age","country","active"],
    table_df=df_easy_ex1,
)

In [41]:
# @title Easy Exercise 2
df_easy_ex2 = pd.DataFrame([
    # TRUE rows
    {"price": 100, "category": "electronics", "stock": 10},
    {"price": 50, "category": "books", "stock": 5},
    {"price": 300, "category": "electronics", "stock": 0},

    # FALSE rows
    {"price": 20, "category": "books", "stock": 0},
    {"price": 400, "category": "clothing", "stock": 10},
    {"price": 80, "category": "books", "stock": 0},
])

make_condition_explorer(
    explorer_id="easy_2",
    title="Easy Exercise 2",
    subtitle="""
Statement

Select products that:

- have price between 50 and 300 inclusive,
- and belong to electronics or books.

Exclude products with stock equal to 0 unless they are electronics.
""",
    default_variables=["price","category","stock"],
    table_df=df_easy_ex2,
)

In [42]:
# @title Easy Exercise 3
df_easy_ex3 = pd.DataFrame([
    # TRUE rows
    {"username": "admin1", "email": "a@gmail.com", "verified": True},
    {"username": "user2", "email": "b@gmail.com", "verified": True},
    {"username": "adminX", "email": "c@yahoo.com", "verified": True},

    # FALSE rows
    {"username": "user3", "email": "d@yahoo.com", "verified": True},
    {"username": "admin4", "email": "e@gmail.com", "verified": False},
    {"username": "test_admin", "email": "f@gmail.com", "verified": True},
])

make_condition_explorer(
    explorer_id="easy_3",
    title="Easy Exercise 3",
    subtitle="""
Statement

Select users who:

- are verified,
- and whose username starts with "admin" or email ends with "@gmail.com".

Exclude usernames that start with "test".
""",
    default_variables=["username","email","verified"],
    table_df=df_easy_ex3,
)

In [43]:

# @title Exercise 1
df_ex1 = pd.DataFrame([
    # TRUE rows
    {"age": 30, "country": "Spain",    "membership": "silver",   "purchases": 4,  "email": "ana@gmail.com",            "suspended": False},
    {"age": 22, "country": "Portugal", "membership": "gold",     "purchases": 1,  "email": "bruno@company.com",       "suspended": False},
    {"age": 55, "country": "France",   "membership": "basic",    "purchases": 12, "email": "claire@yahoo.com",        "suspended": False},

    # FALSE rows
    {"age": 28, "country": "Italy",    "membership": "silver",   "purchases": 2,  "email": "diego@gmail.com",         "suspended": False},
    {"age": 35, "country": "Spain",    "membership": "gold",     "purchases": 5,  "email": "elena@company.com",       "suspended": True},
    {"age": 60, "country": "France",   "membership": "basic",    "purchases": 8,  "email": "testfrancois@gmail.com",  "suspended": False},
])

make_condition_explorer(
    explorer_id="exercise_1",
    title="Exercise 1",
subtitle="""
Statement

Select users who satisfy all of the following:

- they are from Spain, Portugal, or Italy,
- and they are either between 25 and 40 inclusive with at least 3 purchases, or they have **gold** membership and an email ending in **@company.com**,
- and they are not suspended.

However, also allow users from France if they are older than 50 and either have more than 10 purchases or have **platinum** membership.

Exclude any user whose email starts with test unless that user is from Italy and has **gold** or **platinum** membership.
""",
    default_variables= ["age", "country", "membership", "purchases", "email", "suspended"],
    table_df=df_ex1,

)

In [44]:
# @title Exercise 2
df_ex2 = pd.DataFrame([
    # TRUE rows
    {"salary": 90000, "department": "engineering", "years_experience": 6, "performance_score": 88, "remote": False, "manager_title": "Lead"},
    {"salary": 130000, "department": "management",  "years_experience": 10, "performance_score": 70, "remote": True,  "manager_title": "Senior Manager"},
    {"salary": 150000, "department": "management",  "years_experience": 12, "performance_score": 60, "remote": False, "manager_title": "Director"},

    # FALSE rows
    {"salary": 80000, "department": "data", "years_experience": 4, "performance_score": 90, "remote": False, "manager_title": "Analyst"},
    {"salary": 75000, "department": "engineering", "years_experience": 5, "performance_score": 80, "remote": False, "manager_title": "Interim Lead"},
    {"salary": 125000, "department": "management", "years_experience": 9, "performance_score": 85, "remote": False, "manager_title": "Interim Director"},
])

make_condition_explorer(
    explorer_id="exercise_2",
    title="Exercise 2",
    subtitle="""
Select employees who meet one of these business rules:

- they work in engineering or data, have a salary from 70000 to 110000 inclusive, and also meet one of these:
  - at least 5 years of experience and performance score at least 85,
  - or at least 8 years of experience,

- or they work in management, have salary above 120000, and are either remote or have a manager title containing "Director".

Exclude employees in data with fewer than 6 years of experience unless their performance score is above 92 and they are not remote.

Also exclude any employee whose manager title starts with "Interim" unless they work in management and earn more than 140000.
""",
    default_variables=["salary","department","years_experience","performance_score","remote","manager_title"],
    table_df=df_ex2,
)

In [45]:
# @title In-class Exercise 3
df_ex3 = pd.DataFrame([
    # TRUE rows
    {"order_amount": 800, "region": "North", "customer_tier": "gold", "coupon_code": "SAVE10", "items_count": 5, "fraud_flag": False},
    {"order_amount": 2500, "region": "South", "customer_tier": "vip", "coupon_code": "VIP20", "items_count": 2, "fraud_flag": False},
    {"order_amount": 6000, "region": "South", "customer_tier": "vip", "coupon_code": "SAFE", "items_count": 3, "fraud_flag": True},
    # FALSE rows
    {"order_amount": 400, "region": "North", "customer_tier": "basic", "coupon_code": "TEMP50", "items_count": 3, "fraud_flag": False},
    {"order_amount": 3000, "region": "East", "customer_tier": "vip", "coupon_code": "FREE100", "items_count": 5, "fraud_flag": False},
    {"order_amount": 7000, "region": "West", "customer_tier": "gold", "coupon_code": "TEMPX", "items_count": 2, "fraud_flag": True},
])

make_condition_explorer(
    explorer_id="exercise_3",
    title="In class Exercise 3",
    subtitle="""
Select orders that meet one of these conditions:

- region is North or West, amount between 300 and 1500 inclusive, and either at least 4 items or tier is gold,
- or tier is vip, amount above 2000, and region is East or South.

Reject fraud_flag TRUE unless:
- tier is vip and amount above 5000,
- or region is West and coupon_code ends with SAFE.

Reject coupon_code starting with TEMP or containing FREE unless:
- items_count > 10 and (region is South or tier is vip).
""",
    default_variables=["order_amount","region","customer_tier","coupon_code","items_count","fraud_flag"],
    table_df=df_ex3,
)

In [46]:
# @title Exercise 4
df_ex4 = pd.DataFrame([
    # TRUE rows
    {"username": "user1", "age": 30, "country": "Spain", "verified": True, "login_count": 10, "account_type": "pro", "email": "a@mail.com"},
    {"username": "user2", "age": 35, "country": "Canada", "verified": True, "login_count": 2, "account_type": "enterprise", "email": "b@business.org"},
    {"username": "guestX", "age": 28, "country": "Germany", "verified": True, "login_count": 25, "account_type": "basic", "email": "c@mail.com"},

    # FALSE rows
    {"username": "guest1", "age": 30, "country": "Spain", "verified": True, "login_count": 5, "account_type": "pro", "email": "d@mail.com"},
    {"username": "user3", "age": 40, "country": "Spain", "verified": False, "login_count": 10, "account_type": "pro", "email": "e@mail.com"},
    {"username": "user4", "age": 50, "country": "Canada", "verified": False, "login_count": 1, "account_type": "enterprise", "email": "temp@mail.com"},
])

make_condition_explorer(
    explorer_id="exercise_4",
    title="Exercise 4",
    subtitle="""
Statement

Select verified users satisfying:

- (Spain or Germany) AND age 21–45 AND (login_count >=5 OR account_type = pro),
- OR Canada AND account_type = enterprise AND (age >30 OR email ends with @business.org).

Exclude usernames starting with guest unless:
- country is Germany AND (account_type = pro OR login_count >20).

Exclude emails containing temp unless:
- account_type = enterprise AND (country = Canada OR age >40).
""",
    default_variables=["username","age","country","verified","login_count","account_type","email"],
    table_df=df_ex4,
)

In [47]:
# @title Exercise 5
df_ex5 = pd.DataFrame([
    # TRUE rows
    {"transaction_amount": 3000, "account_type": "business", "country": "Spain", "flagged": False, "attempts": 1, "description": "normal", "approved_by": "manager"},
    {"transaction_amount": 20000, "account_type": "corporate", "country": "Germany", "flagged": True, "attempts": 3, "description": "manual review needed", "approved_by": "staff"},
    {"transaction_amount": 35000, "account_type": "corporate", "country": "Netherlands", "flagged": False, "attempts": 1, "description": "test ok", "approved_by": "lead"},

    # FALSE rows
    {"transaction_amount": 2000, "account_type": "business", "country": "Spain", "flagged": True, "attempts": 3, "description": "normal", "approved_by": "manager"},
    {"transaction_amount": 12000, "account_type": "corporate", "country": "Spain", "flagged": False, "attempts": 1, "description": "test run", "approved_by": "tempUser"},
    {"transaction_amount": 4000, "account_type": "business", "country": "Germany", "flagged": False, "attempts": 5, "description": "normal", "approved_by": "manager"},
])

make_condition_explorer(
    explorer_id="exercise_5",
    title="Exercise 5",
    subtitle="""
Statement

Select transactions:

- business or corporate in Spain/Germany/Netherlands AND:
  - amount 1000–8000 AND attempts <=2,
  - OR amount >15000,

- OR Belgium enterprise AND (description contains "urgent" OR approved_by = senior_manager).

Exclude flagged unless:
- corporate AND amount >25000,
- OR Germany AND (description contains "manual review" OR approved_by = risk_lead).

Exclude description starting with test OR approved_by starting with temp unless:
- Netherlands AND amount >30000 AND account_type != business.
""",
    default_variables=["transaction_amount","account_type","country","flagged","attempts","description","approved_by"],
    table_df=df_ex5,
)